In [ ]:
!nvidia-smi

In [ ]:
!ls /content/drive
!ls /content/drive/MyDrive
!ls /content/drive/MyDrive/FYP


In [ ]:
# 2. connect Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. copy the zip to the fast local disk and unzip it (much faster than reading from Drive)
!cp /content/drive/MyDrive/FYP/merged_subset.zip /content/
!unzip -q /content/merged_subset.zip -d /content/data
!ls /content/data

In [ ]:
# 4. install ultralytics
!pip install -q ultralytics

In [ ]:
# 5. write a new data.yaml with the Colab paths (the old one has Windows paths)
import os
import yaml

# if the zip made an extra folder level, point this at the folder that has images/ and labels/
data_root = "/content/data/merged_subset"
if not os.path.exists(data_root):
    data_root = "/content/data"

with open(os.path.join(data_root, "data.yaml")) as f:
    old = yaml.safe_load(f)

new = {
    "path": data_root,
    "train": "images/train",
    "val": "images/valid",
    "test": "images/test",
    "nc": old["nc"],
    "names": old["names"],
}
with open("/content/data.yaml", "w") as f:
    yaml.safe_dump(new, f, sort_keys=False)

print("train images:", len(os.listdir(os.path.join(data_root, "images/train"))))
print("classes:", new["nc"])

In [ ]:
# 6. train. results are saved to Drive, so nothing is lost if Colab disconnects
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(
    data="/content/data.yaml",
    epochs=40,
    imgsz=640,
    batch=32,
    close_mosaic=5,
    workers=2,
    project="/content/drive/MyDrive/FYP/runs",
    name="colab-run4",
)

In [ ]:
!ls /content/drive/MyDrive/FYP/runs
!find /content/drive/MyDrive/FYP/runs -name best.pt

In [ ]:
# 8. final per-class results on the validation set
from ultralytics import YOLO

best = YOLO("/content/drive/MyDrive/FYP/runs/colab-run1-2/weights/best.pt")
metrics = best.val(data="/content/data.yaml", imgsz=640)

Download `best.pt` from `FYP/runs/colab-run1/weights/` in your Drive and put it in your project's `models/` folder.
Send me the table printed by the last cell.

In [ ]:
import os
from ultralytics import YOLO

base = "/content/drive/MyDrive/FYP/runs"
RUN_NAME = "colab-run5-v8s"
last = f"{base}/{RUN_NAME}/weights/last.pt"

if os.path.exists(last):
    # an earlier session was cut off: continue from the last saved epoch
    print("resuming from", last)
    YOLO(last).train(resume=True)
else:
    model = YOLO("yolov8s.pt")          # the only change from run4: s instead of n
    model.train(
        data="/content/data.yaml",
        epochs=40,
        imgsz=640,
        batch=32,
        close_mosaic=5,
        workers=2,
        project=base,
        name=RUN_NAME,
        exist_ok=True,                   # stops Ultralytics making colab-run5-v8s-2
    )

In [ ]:
from ultralytics import YOLO
import pandas as pd

base = "/content/drive/MyDrive/FYP/runs"
RUNS = {"v8n": "colab-run4", "v8s": "colab-run5-v8s"}
DATA = "/content/data.yaml"

results, speed, params = {}, {}, {}
for label, folder in RUNS.items():
    print("evaluating", label, "...")
    m = YOLO(f"{base}/{folder}/weights/best.pt")
    r = m.val(data=DATA, imgsz=640, batch=1, verbose=False, plots=False)
    rows = {"ALL": (r.box.map50, r.box.mr)}
    for i, c in enumerate(r.box.ap_class_index):
        p, rec, ap50, ap = r.box.class_result(i)
        rows[m.names[c]] = (ap50, rec)
    results[label] = rows
    speed[label] = r.speed["inference"]
    params[label] = sum(p.numel() for p in m.model.parameters()) / 1e6

a, b = list(RUNS)
classes = list(results[a].keys())
nan = (float("nan"), float("nan"))
table = pd.DataFrame({
    f"{a} mAP50": [results[a].get(c, nan)[0] for c in classes],
    f"{b} mAP50": [results[b].get(c, nan)[0] for c in classes],
    f"{a} R":     [results[a].get(c, nan)[1] for c in classes],
    f"{b} R":     [results[b].get(c, nan)[1] for c in classes],
}, index=classes).round(3)
table["mAP50 change"] = (table[f"{b} mAP50"] - table[f"{a} mAP50"]).round(3)

table.to_csv(f"{base}/{a}_vs_{b}.csv")
print(table.to_string())
print()
for label in RUNS:
    print(f"{label}: {params[label]:.1f} M parameters, {speed[label]:.1f} ms per image on the T4 (batch 1)")
print("\nsaved to", f"{base}/{a}_vs_{b}.csv")